<a href="https://colab.research.google.com/github/mirsaidl/Face_Recognition/blob/main/roommates_facial_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Face Recognition**

In [1]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import os
import face_recognition
from ultralytics import YOLO
from collections import defaultdict

detector = YOLO("models/yolov8m-face.pt")

In [2]:
database_path = "data/ilhan-org"
database_dict = defaultdict(np.ndarray)

for file_name in os.listdir(database_path):
    img_full_path = os.path.join(database_path, file_name)

    # Load the image
    img = face_recognition.load_image_file(img_full_path)

    encodings = face_recognition.face_encodings(img)[0]

    database_dict[file_name.split(".")[0]] = encodings

### **Face recognition in an image**

In [3]:
# Initialize variables
face_locations = []
face_encodings = []
face_names = []

# Open video capture
video_capture = cv2.VideoCapture('client/pred_videos/videoa1-1_eval.mp4')

# Get the dimensions of the input video
width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the codec and create a VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 20.0, (width, height))

while True:
    # Grab a single frame of video
    ret, frame = video_capture.read()

    if not ret:
        print("No frame captured, exiting...")
        break

    
    detections = detector(frame, conf=0.5, verbose=False)
    
    if len(detections[0].boxes.data) == 0:
        continue
    
    boxes = detections[0].boxes.data.cpu().tolist()

    face_locations = []
    for box in boxes:
        x1, y1, x2, y2, _, _ = map(int, box)
        face_locations.append((y1, x2, y2, x1))  # (top, right, bottom, left)

    #face_locations = [(top, right, bottom, left) for top, right, bottom, left in boxes]
    face_encodings = face_recognition.face_encodings(frame, face_locations, num_jitters=1, model="large")

    face_names = []
    for face_encoding in face_encodings:
        # See if the face is a match for the known face(s)
        matches = face_recognition.compare_faces(list(database_dict.values()), face_encoding, tolerance=0.5)
        name = "Unknown"

        face_distances = face_recognition.face_distance(list(database_dict.values()), face_encoding)

        best_match_index = np.argmin(face_distances)
        if matches[best_match_index]:
            name = list(database_dict.keys())[best_match_index]


        if name != "Unknown":
            print(f"Detected {name}")

        face_names.append(name)

    # Display the results
    for (top, right, bottom, left), name in zip(face_locations, face_names):
        # Draw a box around the face
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 0, 255), 2)

        # Draw a label with a name below the face
        cv2.rectangle(frame, (left, bottom - 20), (right, bottom), (0, 0, 255), cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        
        cv2.putText(frame, name, (left , bottom), font, 1.0, (255, 255, 255), 1)

    # Write the frame with face recognition annotations to the output video
    out.write(frame)

    # Display the resulting image
    cv2.imshow('Video', frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture and writer, and close all windows
video_capture.release()
out.release()
cv2.destroyAllWindows()

Detected Zaripova Sevara
Detected MushtariyTurdaliyeva
Detected DilraboQobilova
Detected DilraboQobilova
Detected DilraboQobilova
Detected DilraboQobilova
Detected DilraboQobilova
Detected Shodiyona Saydullayeva
Detected DilraboQobilova
Detected DilraboQobilova
Detected DilraboQobilova
Detected DilraboQobilova
Detected DilraboQobilova
Detected DilraboQobilova
Detected DilraboQobilova
Detected Zuhra Dolimova
Detected DilraboQobilova
Detected Zuhra Dolimova
Detected DilraboQobilova
Detected DilraboQobilova
Detected Zuhra Dolimova
Detected Zaripova Sevara
Detected Zuhra Dolimova
Detected DilraboQobilova
Detected Zaripova Sevara
Detected DilraboQobilova
Detected Zaripova Sevara
Detected Zuhra Dolimova
Detected Zaripova Sevara
Detected Zuhra Dolimova
Detected Zaripova Sevara
Detected MushtariyTurdaliyeva
Detected GulchexraObidova
Detected GulchexraObidova
Detected GulchexraObidova
Detected GulchexraObidova
Detected GulchexraObidova
Detected GulchexraObidova
Detected GulchexraObidova
Detecte

In [15]:
help(face_recognition.compare_faces)

Help on function compare_faces in module face_recognition.api:

compare_faces(known_face_encodings, face_encoding_to_check, tolerance=0.6)
    Compare a list of face encodings against a candidate encoding to see if they match.
    
    :param known_face_encodings: A list of known face encodings
    :param face_encoding_to_check: A single face encoding to compare against the list
    :param tolerance: How much distance between faces to consider it a match. Lower is more strict. 0.6 is typical best performance.
    :return: A list of True/False values indicating which known_face_encodings match the face encoding to check

